# 🎴 AI 俳句バトル + 匿名相互審査

**遊びの実験**: LLM-jp Playground の 4 モデルに同じお題で俳句を 1 句ずつ詠ませる。
**匿名化**して A/B/C/D とラベルし直してから、再び 4 モデル自身を審査員として呼び戻し、互いに採点させる。

**観察ポイント**:
- 🏆 総合王者は誰か (Borda count)
- 🪞 自分の作品を識別できるか — もし自作だと気付けば 1 位に置くだろうし、気付かなければ普通の順位を付ける
- 🥶 自作に **最下位** を付ける塩対応モデルが出るか
- 🌶️ 審査員ごとの好みの偏り (Gemma は派手目を好む？ Qwen は伝統派？)


## 1. セットアップ — 最小ヘルパ

In [ ]:
import os, re, random, time
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=3000, temperature=0.8,
         max_retries=2):
    """内部 streaming 1 ショット。
    - thinking モデルでも reasoning を落とさない
    - stream 途中で中断したら自動リトライ (LLM-jp 8b/32b thinking で時々発生)
    - 全リトライ失敗時は、最後に集めた content または reasoning を返す
    """
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _create(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    last_reasoning = ""
    for attempt_i in range(max_retries + 1):
        try:
            try:
                stream = _create(True)
            except Exception:
                stream = _create(False)
        except Exception as e:
            if attempt_i < max_retries:
                wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
                print(f"\n  ⚠️  create failed ({type(e).__name__}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                      flush=True)
                time.sleep(wait)
                continue
            return ""
        content, reasoning = [], []
        interrupted = None
        try:
            for chunk in stream:
                if not chunk.choices: continue
                d = chunk.choices[0].delta
                c = getattr(d, "content", None)
                if c: content.append(c)
                for f in ("reasoning_content", "reasoning"):
                    v = getattr(d, f, None)
                    if v: reasoning.append(v); break
        except Exception as e:
            interrupted = type(e).__name__
        text = "".join(content).strip()
        last_reasoning = "".join(reasoning).strip()
        if text:
            return text
        if not interrupted:
            return last_reasoning  # 正常終了で content 空 → reasoning を採用
        # 中断 + content 空 → リトライ
        if attempt_i < max_retries:
            wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
            print(f"\n  ⚠️  stream interrupted ({interrupted}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                  flush=True)
            time.sleep(wait)
            continue
        print(f"\n  ⚠️  stream interrupted ({interrupted}); all retries exhausted",
              flush=True)
        return last_reasoning
    return last_reasoning

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
print("4 モデル準備完了:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. お題で 4 モデルが詠む

In [ ]:
THEME = "深夜の研究室、コーヒーと計算機"

PROMPT = (
    f"お題『{THEME}』で五七五の俳句を **1 句だけ** 詠んでください。\n"
    "- 出力は俳句のみ、**1 行**で。スラッシュ「/」で区切っても可。\n"
    "- 説明・前置き・複数候補・引用符は禁止。\n"
    "- 季語の有無は問いません。"
)

def extract_haiku(raw):
    """出力から最も俳句らしい 1 行を抜き出す簡易フィルタ"""
    lines = [l.strip(" \t-・*「」『』\"'") for l in raw.strip().splitlines()]
    lines = [l for l in lines if l and not l.startswith("#")]
    if not lines: return "(no output)"
    # 「お題に沿った俳句:」みたいな前置きを避けるため、5文字以上15-40文字程度の行を優先
    candidates = [l for l in lines if 8 <= len(l) <= 60]
    return candidates[-1] if candidates else lines[-1]

print(f"🎴 お題: 「{THEME}」\n")
haiku_pool = []
for name, mid in VOICES.items():
    print(f"  ✍️  {name} ...", end=" ", flush=True)
    raw = chat(mid, PROMPT, temperature=0.9, max_tokens=2500)
    haiku = extract_haiku(raw)
    haiku_pool.append({"poet": name, "model": mid, "haiku": haiku, "raw": raw})
    print(haiku)


## 3. 匿名化シャッフル — 誰がどれか、本人たちには内緒

In [ ]:
random.seed()  # 毎回違うシャッフル
labels = ["A", "B", "C", "D"]
order = list(range(4))
random.shuffle(order)

anonymized = []
for label, idx in zip(labels, order):
    anonymized.append({
        "label": label,
        "haiku": haiku_pool[idx]["haiku"],
        "true_poet": haiku_pool[idx]["poet"],
    })

display(Markdown(
    "### 🎭 匿名化された 4 句\n\n" +
    "\n\n".join(f"**{x['label']}** ─ {x['haiku']}" for x in anonymized)
))


## 4. 互いに審査 — 4 モデルが全句に順位を付ける

In [ ]:
JUDGE_PROMPT = (
    f"以下は同じお題『{THEME}』で詠まれた 4 つの俳句です。\n"
    "芸術性・お題への沿い方・余韻の観点から 1 位〜 4 位の順位を付けてください。\n\n"
    f"A: {anonymized[0]['haiku']}\n"
    f"B: {anonymized[1]['haiku']}\n"
    f"C: {anonymized[2]['haiku']}\n"
    f"D: {anonymized[3]['haiku']}\n\n"
    "回答は **厳密に** 以下の形式で出力してください:\n\n"
    "1位: X\n2位: X\n3位: X\n4位: X\n講評: (2 行以内)\n\n"
    "X には A, B, C, D のいずれかを入れる。同順位や説明文は禁止。"
)

RANK_PAT = re.compile(r'([1-4１-４一二三四])\s*位\s*[:：]?\s*([ABCD])', re.I)
ZEN2HAN = str.maketrans("１２３４一二三四", "12341234")

def parse_ranks(text):
    ranks = {}
    for m in RANK_PAT.finditer(text.translate(ZEN2HAN)):
        r = int(m.group(1))
        if r not in ranks:
            ranks[r] = m.group(2).upper()
    return ranks

print("🧑‍⚖️ 審査員を巡回中...\n")
judgments = []
for name, mid in VOICES.items():
    print(f"  審査: {name} ...", end=" ", flush=True)
    resp = chat(mid, JUDGE_PROMPT, temperature=0.0, max_tokens=3500)
    ranks = parse_ranks(resp)
    judgments.append({"judge": name, "ranks": ranks, "raw": resp})
    if len(ranks) == 4 and len(set(ranks.values())) == 4:
        print(" > ".join(ranks[i] for i in [1,2,3,4]))
    else:
        print(f"⚠️ パース部分成功 ({ranks})")


## 5. 🏆 結果発表 + 🪞 自作評価分析

In [ ]:
# Borda count: 1 位 = 4 点 ... 4 位 = 1 点
scores = {x["label"]: 0 for x in anonymized}
for j in judgments:
    for r, lab in j["ranks"].items():
        if lab in scores:
            scores[lab] += (5 - r)

ranked = sorted(scores.items(), key=lambda x: -x[1])
MEDALS = ["🥇", "🥈", "🥉", "🍀"]

rows = []
for i, (label, score) in enumerate(ranked):
    item = next(x for x in anonymized if x["label"] == label)
    rows.append(
        f"### {MEDALS[i]} {label} ─ {item['true_poet']} ({score} 点)\n\n"
        f"> {item['haiku']}"
    )
display(Markdown("## 🏆 総合順位\n\n" + "\n\n".join(rows)))

# 自作評価
label_of = {x["true_poet"]: x["label"] for x in anonymized}
mirror = []
for j in judgments:
    own_label = label_of[j["judge"]]
    own_rank = next((r for r, lab in j["ranks"].items() if lab == own_label), None)
    if own_rank is None:
        mirror.append(f"❓ **{j['judge']}** ─ 自作 ({own_label}) のランクをパース失敗")
        continue
    face = {1: "😎 自画自賛", 2: "😊 控えめに 2 位", 3: "😅 自分を 3 位", 4: "🥶 自作に最下位"}[own_rank]
    mirror.append(f"- **{j['judge']}** → 自作 ({own_label}) を **{own_rank} 位** に評価 ({face})")

display(Markdown("## 🪞 各審査員、自作をどう評価したか\n\n" + "\n".join(mirror)))


## おまけ

- 結果を見ると、**自作識別はほぼ不可能**であることが多い (Verifier に同モデルを使う際の安心材料)
- 一方で **審査員ごとの好みの偏り** はわりと安定する (Borda 順位を seed を変えて何度か走らせると傾向が見える)
- お題を `"古代の図書館、本の埃と燭台"` などに変えて再実行してみてください。お題への適応力が見えて面白いです
- 教育用途: GVR パイプライン (`02_gvr_pipeline.ipynb`) で Verifier に複数モデルを使う妥当性の傍証としても使えます
